In [ ]:
import warnings
warnings.filterwarnings('ignore')
from requests import get
from requests.exceptions import RequestException
from contextlib import closing
from bs4 import BeautifulSoup
import json
import requests
print(requests.__version__)
import csv
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import statistics
import random
%matplotlib inline

%run db.py

In [ ]:
%run db.py

In [4]:
def get_ratings_all(start_date,end_date,superlike_weight=3):
    """
    Load all user ratings from the database.

    Rating scale:
    - Super-like (seq=2): superlike_weight points
    - Like (seq=0): 1 point
    - Dislike: -1 points

    Returns:
        DataFrame with columns: timestamp, user_id, song_id, rating,
        plus song metadata (title, artist, genre, etc.)
    """
    sql = """
        SELECT * FROM (
            -- Super-likes: 5 points
            SELECT us.timestamp, us.user_id, song_id, %s as rating
            FROM user_songs us
            WHERE seq = 2
            AND   us.timestamp>='%s' and us.timestamp<'%s'
            UNION

            -- Regular likes: 1 point
            SELECT us.timestamp, us.user_id, song_id, 1 as rating
            FROM user_songs us
            WHERE seq = 0
            AND   us.timestamp>='%s' and us.timestamp<'%s'
            UNION

            -- Dislikes: -1 points
            SELECT timestamp, us.user_id, song_id, -1 as rating
            FROM unliked_songs us
            WHERE   us.timestamp>='%s' and us.timestamp<'%s'
        ) t
        INNER JOIN song_ids s ON t.song_id = s.song_id
        INNER JOIN catalog c ON t.song_id = c.id
        LEFT JOIN artist_genre ag ON c.artist_id = ag.artist_id
        WHERE piki_video_url IS NOT NULL
        ORDER BY t.timestamp ASC
    """%(superlike_weight,start_date,end_date,start_date,end_date,start_date,end_date)

    res = DB.fetch(sql)
    df = pd.DataFrame(res)

    print(f"✓ Loaded {len(df):,} ratings")
    print(f"  - Users: {df['user_id'].nunique():,}")
    print(f"  - Songs: {df['song_id'].nunique():,}")

    return df

In [ ]:
ratings=get_ratings_all('2021-01-01','2027-01-01',10)

In [6]:
def adjust_ratings(ratings,min_ratings_user,min_ratings_item,item_reg,learning_rate):
    item_rating_avg=ratings[['user_id','song_id','rating']].groupby('song_id').mean()
    item_rating_count=ratings[['user_id','song_id','rating']].groupby('song_id').count()
    # regularize items
    item_rating_avg['rating']=(item_rating_avg['rating']*item_rating_count['rating']+0.5*item_reg)/(item_rating_count['rating']+item_reg)
    df_items=item_rating_avg[item_rating_count['user_id']>=min_ratings_item]['rating'].rename('item_rating')
    ratings_with_item_means = ratings.merge(df_items, 
           how = 'inner',
           left_on = 'song_id',
           right_on = 'song_id')
    ratings_with_item_means['stringency']=ratings_with_item_means['item_rating']-ratings_with_item_means['rating']
    user_stringency_avg=ratings_with_item_means[['user_id','stringency']].groupby('user_id').mean()
    user_stringency_cnt=ratings_with_item_means[['user_id','stringency']].groupby('user_id').count()
    df_users=user_stringency_avg[user_stringency_cnt['stringency']>=min_ratings_user]
    ratings_with_stringency = ratings.merge(df_users, 
           how = 'inner',
           left_on = 'user_id',
           right_on = 'user_id')
    ratings_with_stringency['adj rating']=ratings_with_stringency['rating']+learning_rate*ratings_with_stringency['stringency']
    ratings_with_stringency['rating']=ratings_with_stringency['adj rating']
    ratings_with_stringency=ratings_with_stringency.drop('stringency', axis=1)
    ratings_with_stringency=ratings_with_stringency.drop('adj rating', axis=1)
    ratings_with_stringency=ratings_with_stringency[ratings_with_stringency['rating'].notna()]
    ratings_with_stringency=ratings_with_stringency[ratings_with_stringency['song_id'].isin(df_items.index)]
    return ratings_with_stringency ,df_items,df_users

In [7]:
def get_genre_list():
    sql='''select genre,count(*) as cnt from artist_genre group by genre order by cnt desc;'''
    rows=DB.fetch(sql)
    return rows

def update_dislikes():
    sql='''update piki_score 
    join
    (select song_id,count(*) as num_dislikes from unliked_songs  us 
     group by song_id ) as l
    on piki_score.song_id=l.song_id
    set piki_score.num_dislikes=l.num_dislikes
;'''
    DB.execute(sql)

def update_likes():
    sql='''update piki_score 
join
(select song_id,count(*) as num_likes from user_songs  us 
     where seq=0 group by song_id) as l
on piki_score.song_id=l.song_id
set piki_score.num_likes=l.num_likes'''
    DB.execute(sql)

def update_superlikes():
    sql='''update piki_score 
join
(select song_id,count(*) as num_superlikes from user_songs   us 
 where seq=2 group by song_id) as s
on piki_score.song_id=s.song_id
set piki_score.num_superlikes=s.num_superlikes'''
    DB.execute(sql)
    
def update_nulls():
    sql='''update piki_score set num_likes=0 where num_likes is null;'''
    DB.execute(sql)
    sql='''update piki_score set num_superlikes=0 where num_superlikes is null;'''
    DB.execute(sql)
    sql='''update piki_score set num_dislikes=0 where num_dislikes is null;'''
    DB.execute(sql)
    
def update_artists():
    sql='''update piki_score ps inner join catalog c 
on ps.song_id=c.id set ps.artist_id=c.artist_id;'''
    DB.execute(sql)

In [8]:
def delete_investible_songs():
    sql='delete from investible_songs'
    DB.execute(sql)
    
def insert_investible_songs_expectation():
    sql='''insert into investible_songs
select ps.song_id, #artist,title,
greatest( least(round(10*(ifnull(num_likes,0)+ifnull(num_superlikes,0)+5)/(ifnull(num_likes,0)+ifnull(num_superlikes,0)+ifnull(num_dislikes,0)+10), 1) ,8),2)  as cost
,greatest( least(round(-0.45+10*(num_likes+num_superlikes+5)/(num_likes+num_superlikes+num_dislikes+10), 0) ,8),2)  as cost_adj
from piki_score ps 
inner join song_ids s on ps.song_id=s.song_id
group by ps.song_id
order by cost desc;'''
    DB.execute(sql)

In [9]:
item_reg=0
learning_rate=1
min_ratings_user=0
min_ratings_item=0
for i in range(0,15):
    prev_ratings=ratings.copy(deep=True)
    ratings,df_items,df_users=adjust_ratings(ratings,min_ratings_user,min_ratings_item,item_reg,learning_rate)
    print(len(ratings))
    #print(ratings['rating'].mean())
    

214279
214279
214279
214279
214279
214279
214279
214279
214279
214279
214279
214279
214279
214279
214279


In [10]:
print(((ratings['rating']-prev_ratings['rating'])**2).max())

1.1340675837074435e-07


In [48]:
ratings_list=   [0,   2,  5, 10,20,50, 200,10000]
max_rating_list=[94, 95, 96, 97,98,99,100]

In [57]:
delete_piki_score_table()

In [51]:
for i in range(0,7):
    lower_bound=ratings_list[i]
    upper_bound=ratings_list[i+1]
    print(lower_bound)
    print(upper_bound)
    print('***')
    df_items_tmp=item_rating_avg[(item_rating_count['rating']<upper_bound) & (item_rating_count['rating']>=lower_bound)] #['rating']
    a=pd.qcut(df_items_tmp.rating,np.linspace(0,1,101), duplicates='drop')
    length=len(a.cat.categories)
    premium= max_rating_list[i]-length
    df_items_tmp['piki_score'] =(pd.qcut(df_items_tmp.rating, np.linspace(0,1,101),labels=(premium+np.linspace(1,length,length)) , duplicates='drop')) #.sort_values()
    df_items_tmp=df_items_tmp.reset_index()[['song_id','piki_score']] 
    df_items_tmp.to_sql('piki_score', con=engine, if_exists='append', index=False)

0
2
***
2
5
***
5
10
***
10
20
***
20
50
***
50
200
***
200
10000
***


In [59]:
update_dislikes()
update_likes()
update_superlikes()
update_nulls()


In [60]:
update_artists()

In [61]:
delete_investible_songs()
insert_investible_songs_expectation()